In [2]:
import spacy
from scispacy.linking import EntityLinker
from scispacy.abbreviation import AbbreviationDetector
import os
import json
import pandas as pd
import re
import requests
from periodictable import elements
from pathlib import Path
from pypdf import PdfReader as pr
#!pip install nltk
from nltk.tokenize import sent_tokenize
import pubchempy as pcp

import warnings
warnings.filterwarnings("ignore")

In [3]:
#os.chdir('..\\dissertation DLC content')
#linker = Ontology('OntoBiotope_BioNLP-OST-2019.obo')

path_to_scispacy_model = os.path.normpath('C:\\Users\\jace\\Documents\\assignments\\dissertation DLC content\\en_ner_bionlp13cg_md-0.5.4\\en_ner_bionlp13cg_md\\en_ner_bionlp13cg_md-0.5.4')
path_to_microbial_model = os.path.normpath('C:\\Users\\jace\\Documents\\assignments\\dissertation DLC content\\custom_microbial_ner')

#scispacy_model = spacy.load(path_to_scispacy_model)
microbial_chemical_model = spacy.load(path_to_microbial_model)

""" print('Reading .obo file')
onto = obonet.read_obo(obo_path)

kb = KnowledgeBase(get_entities_from_obonet(onto))

linker = scispacy_model.add_pipe('scispacy_linker')
linker.set_kb(kb) """

print('Adding entity linker')
linker = microbial_chemical_model.add_pipe('scispacy_linker', config={'resolve_abbreviations': True, 'linker_name': 'mesh'})

#microbial_model.add_pipe('abbreviation_detector', last=True, validate=False)

Adding entity linker


# <font size=4>hidden</font>

In [4]:
%%script False

print(scispacy_model.pipe_names)

#linker = pyobo.from_obo_path('..\\dissertation DLC content\\chebi_lite.obo', version='1.2')
kb = KnowledgeBase('..\\dissertation DLC content\\chebi_lite.json')

linker = scispacy_model.add_pipe('entity_linker', after='ner')
linker.set_kb(kb)

onto = pyobo.fr('chebi_core')

#update this here to try and download a more relevant knowledge base
""" !python -m spacy_entity_linker "download_knowledge_base"

from spacy.kb import InMemoryLookupKB

def create_kb(vocab):
    kb = InMemoryLookupKB(vocab, entity_vector_length=128)
    return kb

linker = scispacy_model.add_pipe('entityLinker', after='ner') """

Couldn't find program: 'False'


# <font size=4>not hidden</font>

In [5]:
with open('list_files\\utilizations_list.json', 'r', encoding='utf-8') as f:
    utilizations = list(json.load(f))

basic_utilizations = ['fermentation', 'hydrolyze', 'hydrolyse', 'reduce', 'reduces', 'reduced', 'produce', 'production', 'produces', 'consume', 'consumes', 'consumption', 'consuming', 'producing', 'grow', 'growth', 'oxidise', 'oxidize', 'consumed', 'produced', 'convert', 'converts', 'converting', 'converted', 'conversion']

print('|'.join(basic_utilizations))

util_mappings = {
    'production': 'PRODUCES',
    'produce': 'PRODUCES',
    'produces': 'PRODUCES'
}
    
custom_labels = ['MICROBE_NAME', 'SIMPLE_CHEMICAL', 'CHEMICAL']

with open('list_files\\genus_names.json', 'r', encoding='utf-8') as f:
    genera_unfiltered = list(json.load(f))
    
with open('list_files\\food_list.json', 'r', encoding='utf-8') as f:
    food_list = list(json.load(f))
    
genus_names = []

for gen in genera_unfiltered:
    if 'Candidatus' in gen:
        gen_split = gen.split(' ')
        genus_names.append(gen_split[1])
    else:
        genus_names.append(gen)
        
with open('list_files\\microbes_genus_species.json', 'r', encoding='utf-8') as f:
    microbes_genus_species = list(json.load(f))
    
not_chemicals = ['flavour', 'flavor', 'flavours', 'flavors', 'carbohydrate', 'carbohydrates', 'vitamin', 'vitamins', 'aroma', 'aromas', 'flavonoid', 'flavonoids'] + genus_names + food_list
print(len(not_chemicals))
    
g_species = []
genus_species = []
for gs in microbes_genus_species:
    genus, species = gs.split(' ')
    g_species.append(str(genus[0] + '. ' + species))
    genus_species.append((genus, species))
    
print(len(g_species), len(set(g_species)))
    
with open('list_files\\species_names.json', 'r', encoding='utf-8') as f:
    species_names = list(set(json.load(f)))
    
with open('list_files\\basionym_realname_mapping.json', 'r', encoding='utf-8') as f:
    basionym_realname_mapping = dict(json.load(f))
    
print(list(basionym_realname_mapping)[:50])
    
microbe_shortforms_known_mappings = {
    'S. cerevisiae': 'Saccharomyces cerevisiae',
    'A. niger': 'Aspergillus niger',
    'S. fragilis': 'Saccharomyces fragilis',
    'L. hilgardii': 'Lentilactobacillus hilgardii'
}

chemical_mapping = {}

fermentation|hydrolyze|hydrolyse|reduce|reduces|reduced|produce|production|produces|consume|consumes|consumption|consuming|producing|grow|growth|oxidise|oxidize|consumed|produced|convert|converts|converting|converted|conversion
14931
24692 21518
['Microcyclus aquaticus', 'Flectobacillus glomeratus', 'Microcyclus marinus', 'Cellvibrio gilvus', 'Planctomyces brasiliensis', 'Planctomyces maris', 'Borrelia burgdorferi', 'Spirochaeta hermsi', 'Spirochaeta bajacaliforniensis', 'Spirochaeta stenostrepta', 'Spirochaeta thermophila', 'Spirochaeta zuelzerae', 'Treponema hyodysenteriae', 'Spirochaeta pallida', 'Spirochaeta biflexa', 'Spirochaeta interrogans', 'Aquaspirillum magnetotacticum', 'Pelobacter carbinolicus', 'Spirillum lipoferum', 'Wolinella curva', 'Wolinella recta', 'Vibrio sputorum', 'Campylobacter fennelliae', 'Campylobacter pylori', 'Alteromonas colwelliana', 'Bacterium abortus', 'Alteromonas putrefaciens', 'Flavobacterium okeanokoites', 'Flavobacterium balustinum', 'Alteromonas ha

In [ ]:
for ind, file in enumerate(os.listdir("..\\dissertation DLC content\\fermentation_papers_preprocessed")[:3]):
    microbe_shortform_mappings = {}
    print(ind)
    #keep track of longterm names of all microbes seen so far
    filename = os.fsdecode(file)
    if filename.endswith('json'):
        with open(f"..\\dissertation DLC content\\fermentation_papers_preprocessed\\{filename}", 'r') as f:
            text = f.read()
    elif filename.endswith('pdf'):
        try:
            print('pdf!')
            path = Path(os.getcwd() + "\\fermentation_papers\\" + filename)
            reader = pr(path)
            text = ""
            for page in reader.pages:
                text = text + page.extract_text()
        except:
            continue
    else:
        continue
    
    def resolve_basionym(name):
        resolved_name = name
        basionym_mapping_name = basionym_realname_mapping.get(name)
        if basionym_mapping_name:
            resolved_name = basionym_mapping_name
        else:
            results = requests.get(f"https://api.gbif.org/v1/species/search?q={str(resolved_name.replace(' ', '+'))}").json().get('results')
            if results:
                gbif_name = results[0].get('species')
                if gbif_name:
                    resolved_name = gbif_name
        return resolved_name
    
    #do a first pass of the paper to identify microbe shortforms
    all_paper_ents = microbial_chemical_model(text)
    all_mic_ents = [(mic.label_, mic.text, mic.start_char, mic.end_char) for mic in all_paper_ents.ents if mic.label_ == 'MICROBE_NAME']
    all_mic_shapes_raw = [mic.shape_ for mic in all_paper_ents]
    mic_shapes = [(name, ' '.join([msr[:4] for msr in all_mic_shapes_raw[s:e]]), sc, ec) for s, e, name, sc, ec in [(m.start, m.end, m.text, m.start_char, m.end_char) for m in all_paper_ents.ents if m.label_ == 'MICROBE_NAME']]
    for name, shape, s, e in mic_shapes:
        if shape == 'Xxxx xxxx':
            genus, species = name.split(' ')
            shortform_name = genus[0] + '. ' + species
            if not microbe_shortform_mappings.get(shortform_name):
                #normalize microbe name before adding
                #TODO: find alternative name, rather than just 'species'
                resolved_basionym = resolve_basionym(name)
                if resolve_basionym:
                    microbe_shortform_mappings[shortform_name] = resolved_basionym

        elif shape == 'X. xxxx' or shape == 'Xx. xxxx' or shape == 'Xxx. xxxx':
            if not microbe_shortform_mappings.get(name):
                #add that entry in to indicate we have seen the shortform before the longform, in case the shortform doesnt get seen again, sometimes the microbe names are weird
                microbe_shortform_mappings[name] = None
                
    #keep track of previous sentence and entities for which to merge with, if there are not enough to form relations with
    merge_sentence = None
    merge_ents = []
    merge_ents_resolved = []
    
    sentences = sent_tokenize(text)
    for sent in sentences:
        all_ents_raw = microbial_chemical_model(sent)
        mic_ents_unfiltered = [(mic.label_, mic.text, mic.start_char, mic.end_char) for mic in all_ents_raw.ents if mic.label_ == 'MICROBE_NAME']

        #these are filtered but unresolved, so we keep the original, un-normalized names of the microbes, but they are filtered. this is done to remove people's names which
        #are misidentified as microbes
        mic_ents_filtered = []
        mic_shapes_raw = [mic.shape_ for mic in all_ents_raw]
        mic_shapes = [(name, ' '.join([msr[:4] for msr in mic_shapes_raw[s:e]]), sc, ec) for s, e, name, sc, ec in [(m.start, m.end, m.text, m.start_char, m.end_char) for m in all_ents_raw.ents if m.label_ == 'MICROBE_NAME']]
        mic_ents_resolved = []
        
        for name, shape, s, e in mic_shapes:
            if shape == 'X. xxxx' or shape == 'Xx. xxxx' or shape == 'Xxx. xxxx':
                alternate_name = None
                if shape == 'Xxx. xxxx':
                    print("weird name:", name)
                #check the map to see if theres anything in there
                shortform_match_name = microbe_shortform_mappings.get(name)
                #add the resolved name to the array
                if shortform_match_name:
                    resolved_name = resolve_basionym(shortform_match_name)
                    alternate_name = resolved_name
                    mic_ents_resolved.append(('MICROBE_NAME', resolved_name, s, e))
                else:
                    #try with the known mappings (obviously not gonna do them all, just the most common ones)
                    known_name = microbe_shortforms_known_mappings.get(name)
                    if known_name:
                        alternate_name = known_name
                        microbe_shortform_mappings[name] = known_name
                        mic_ents_resolved.append(('MICROBE_NAME', known_name, s, e))
                    else:
                        #perhaps match against the list of genera and species to see which is the most probable? like start with the first letter and figure shit out from there
                        #like check the species name and check it against the full names, and see if theres any genera which match up
                        #but what if there are multiple candidates?
                        
                        full_genus, full_species = name.split(' ') #genus is first letter, species is 'xxxx' bit after
                        g_species = full_genus[0] + '. ' + full_species
                        #originally dont match with genus to check for potential name changes
                        matched_species = [(gen, spec) for gen, spec in genus_species if spec == full_species]
                        
                        matched_name = None
                        
                        if len(matched_species) == 1:
                            matched_name = matched_species[0]
                        elif len(matched_species) == 2:
                            gen1, spec1 = matched_species[0]
                            resolved_basionym_1 = resolve_basionym(str(gen1 + '+' + spec1))
                            
                            gen2, spec2 = matched_species[1]
                            resolved_basionym_2 = resolve_basionym(str(gen2 + '+' + spec2))
                            
                            print("two shared-species names:", resolved_basionym_1, resolved_basionym_2)
                            
                            if resolved_basionym_1 == resolved_basionym_2:
                                matched_name = (gen1, spec1) #found match!
                                microbe_shortform_mappings[name] = gen1 + '+' + spec1
                            else:
                                #TODO: figure something out idk
                                pass
                        else:
                            #match with genus name rather than just species
                            matched_genus = [(gen, spec) for gen, spec in matched_species if gen[0] == full_genus[0]]
                            if len(set(matched_genus)) == 1:
                                matched_name = matched_genus[0] #found match!
                            else:
                                #figure something out idk, maybe the same thing as above
                                pass
                                
                        alternate_name = matched_name
                               
                        #resolve name and add mapping 
                        if matched_name:
                            resolved_basionym = resolve_basionym(' '.join(matched_name))
                            mic_ents_resolved.append(('MICROBE_NAME', str(resolved_basionym), s, e))
                            #only add the original name to the shortform mapping, not the resolved names
                            microbe_shortform_mappings[name] = matched_name
                            
                        #TODO: consider cases such as 'Lpb. plantarum' - IN PROGRESS
                        #TODO: resolve the 'Lactobacillus curvatus' case
                        
                #sometimes an entity is flagged as a microbe when it is actually not. however, i want to keep the original microbe names for the relation extraction part-
                #the llm will see the context sentence and if we give it the resolved entities it will not know how to match them.
                #so only add to the filtered list if an alternate, resolved name exists.
                if alternate_name:
                    mic_ents_filtered.append(('MICROBE_NAME', name, s, e))
                        
            elif shape == 'Xxxx xxxx':
                resolved_basionym = resolve_basionym(name)
                print('resolved:', resolved_basionym)
                name_split = name.split(' ')
                shortform_name = name_split[0][0] + '. ' + name_split[1]
                #then add it to 'resolved'
                mic_ents_resolved.append(('MICROBE_NAME', resolved_basionym, s, e))
                mic_ents_filtered.append(('MICROBE_NAME', name, s, e))
        """ if mic_ents != mic_ents_resolved:
            print()
            print("microbial mismatch (potential resolve)")
            print(sent[:300])
            print(mic_ents)
            print("resolved (?)")
            print(mic_ents_resolved) """

        chem_ents = []
        chem_ents_resolved = []
        
        for sci in all_ents_raw.ents:
            sci_text = sci.text.lower()
            if 'CHEMICAL' in sci.label_ and sci_text not in not_chemicals + [text.lower() for l, text, s, e in mic_ents_filtered]:
                #only add original chemical ents if they are proven to actually be chemicals - i.e. has a result
                if chemical_mapping.get(sci_text):
                    chem_ents_resolved.append(('CHEMICAL', chemical_mapping[sci_text], sci.start_char, sci.end_char))
                    chem_ents.append(('CHEMICAL', sci_text, sci.start_char, sci.end_char))
                else:
                    comps = pcp.get_compounds(sci_text, 'name')
                    if comps:
                        chem_ents.append(('CHEMICAL', sci_text, sci.start_char, sci.end_char))
                        if not chemical_mapping.get(sci_text):
                            chemical_mapping[sci_text] = comps[0].synonyms[0].lower()
                        chem_ents_resolved.append(('CHEMICAL', comps[0].synonyms[0].lower(), sci.start_char, sci.end_char))
                    #else do nothing since if there are no results then it must not be a chemical

        #chemicals should NOT be less than 3 characters long
        utils_found = re.finditer('|'.join(utilizations), str(sent).lower())
        utilization_ents = [('UTILIZATION', sent[util.start(0):util.end(0)].lower(), util.start(0), util.end(0)) for util in utils_found]
        
        ents_and_labels = [(label, text, start, end) for label, text, start, end in mic_ents_filtered + chem_ents if label in custom_labels and len(text)>=3] + utilization_ents
        ents_and_labels = sorted(ents_and_labels, key= lambda tup: tup[2])
        
        ents_and_labels_resolved = [(label, text, start, end) for label, text, start, end in mic_ents_resolved + chem_ents_resolved if label in custom_labels and text not in not_chemicals and len(text)>=3] + utilization_ents
        ents_and_labels_resolved = sorted(ents_and_labels, key= lambda tup: tup[2])
        
        #make sure there are enough entities to form relations with, maybe combine with another sentence otherwise
        #also check for different labels
        if len(ents_and_labels) >= 3 and len(set([label for label, t, s, e in ents_and_labels])) >= 2:
            if merge_sentence:
                sent = merge_sentence + ' ' + sent
                ents_and_labels = merge_ents + ents_and_labels
                ents_and_labels_resolved = merge_ents_resolved + ents_and_labels_resolved
                
            print(sent[:500])
            print(ents_and_labels)
            
            writable_ents = [(label, text) for label, text, s, e in ents_and_labels]
            
            with open('list_files\\all_paper_entities.jsonl', 'a', encoding='utf-16') as f:
                json.dump((sent, writable_ents), f)
                f.write('\n')
                
            merge_sentence = None
            merge_ents = []
            
        else:
            #if there is already a sentence awaiting merging, and merge ents are not empty
            if merge_sentence and merge_ents:
                merge_sentence = merge_sentence + ' ' + sent
                merge_ents = merge_ents + ents_and_labels
            else:
                if ents_and_labels:
                    merge_sentence = sent
                    merge_ents = ents_and_labels
                
        """ sci_lents = linker(sci_ents)
        linker_labels = []
        for ent in sci_ents.ents:
            #print([linker.kb.cui_to_entity[lent[0]] for lent in ent._.kb_ents])
            if 'CHEMICAL' in ent.label_ and ent.text.lower() not in not_chemicals + [ent.text.lower() for l, text, s, e in mic_ents]:
                if ent._.kb_ents:
                    #print(ent._.kb_ents[0])
                    label = linker.kb.cui_to_entity[ent._.kb_ents[0][0]]
                    linker_labels.append(label.canonical_name)
                else:
                    linker_labels.append(ent.text) 
                    
        linked_sci_ents = [lent for lent in sci_lents.ents if 'CHEMICAL' in lent.label_] """
    print()

0
resolved: Gluconobacter oxydans
Alcohol: immobilized cell membrane of Gluconobacter oxydans in calcium alginate containing PQQ (pyrrolo-quinoline quinone), coated with a nitrocellulose layer; ethanol concentrations of up to 20mg l−1 can be detected.
[('MICROBE_NAME', 'Gluconobacter oxydans', 38, 59)]
[('CHEMICAL', 'alcohol', 0, 7), ('MICROBE_NAME', 'Gluconobacter oxydans', 38, 59), ('CHEMICAL', 'calcium', 63, 70), ('CHEMICAL', 'pqq', 91, 94), ('CHEMICAL', 'pyrrolo-quinoline quinone', 96, 121), ('CHEMICAL', 'ethanol', 160, 167)]
Stanbury A. Whitaker S.J.
[]
[]

1
resolved: Lactococcus lactis
Most lactic acid bacterial isolates from wine would have malolactic conversion capabilities; many strains of Lactococcus lactis, the modern classification of several species of Streptococcus have this capability.
[('MICROBE_NAME', 'Lactococcus lactis', 109, 127)]
[('CHEMICAL', 'lactic acid', 5, 16), ('UTILIZATION', 'conversion', 68, 78), ('MICROBE_NAME', 'Lactococcus lactis', 109, 127)]
resolved: 

KeyboardInterrupt: 